In [4]:
!pip install -q langchain langchain-core langchain-community pydantic duckduckgo-search ddgs langchain-experimental requests langchain-huggingface

In [2]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [11]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [6]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """ Return the product of two given numbers a and b"""
  return a*b

In [7]:
multiply.invoke({'a':3, 'b':4})

12

In [8]:
multiply.name

'multiply'

In [9]:
multiply.description

'Return the product of two given numbers a and b'

In [10]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [12]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm)

In [13]:
# tool binding
llm_with_tools = llm.bind_tools([multiply])

In [40]:
query=HumanMessage("Can you multiply 3 with 15 ")

In [41]:
messages=[query]
messages

[HumanMessage(content='Can you multiply 3 with 15 ', additional_kwargs={}, response_metadata={})]

In [42]:
result=llm_with_tools.invoke(messages)

In [43]:
messages.append(result)
messages

[HumanMessage(content='Can you multiply 3 with 15 ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":15}', 'name': 'multiply', 'description': None}, 'id': 'call_uopgomckwgdrkd99leshzpld', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 219, 'total_tokens': 245}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb887-ed7e-7010-9564-501f7f589634-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 15}, 'id': 'call_uopgomckwgdrkd99leshzpld', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 26, 'total_tokens': 245})]

In [44]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 15},
  'id': 'call_uopgomckwgdrkd99leshzpld',
  'type': 'tool_call'}]

In [45]:
# tool execution
multiply.invoke(result.tool_calls[0]['args'])

45

In [46]:
# tool execution
tool_result=multiply.invoke(result.tool_calls[0])

In [47]:
messages.append(tool_result)
messages

[HumanMessage(content='Can you multiply 3 with 15 ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":15}', 'name': 'multiply', 'description': None}, 'id': 'call_uopgomckwgdrkd99leshzpld', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 219, 'total_tokens': 245}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb887-ed7e-7010-9564-501f7f589634-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 15}, 'id': 'call_uopgomckwgdrkd99leshzpld', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 26, 'total_tokens': 245}),
 ToolMessage(content='45', name='multiply', tool_call_id='call_uopgomckwgdrkd99leshzpld')]

In [48]:
llm_with_tools.invoke(messages).content

'Sure, the multiplication of 3 and 15 is 45.'